In [8]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/marketing_AB_clean.csv")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 588,101
Columns: 6


In [9]:
import sys
from pathlib import Path

# Make project-level src modules importable when the notebook is run from notebooks/.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.statistics import conversion_summary, two_proportion_test, cohens_h
from src.validation import (
    check_group_integrity,
    check_user_uniqueness,
    check_conversion_integrity,
    calculate_srm,
)
experiment_summary = conversion_summary(df)

conversion_test = two_proportion_test(df)
ad_rate = conversion_test["treatment_rate"]
psa_rate = conversion_test["control_rate"]
absolute_difference = conversion_test["absolute_difference"]
relative_lift = conversion_test["relative_lift"]

print(experiment_summary)
print(f"\nAd conversion rate : {ad_rate:.4%}")
print(f"PSA conversion rate: {psa_rate:.4%}")
print(f"Absolute lift      : {absolute_difference:.4%}")
print(f"Relative lift      : {relative_lift:.2%}")


  test group   users  conversions  conversion_rate
0         ad  564577        14423         0.025547
1        psa   23524          420         0.017854

Ad conversion rate : 2.5547%
PSA conversion rate: 1.7854%
Absolute lift      : 0.7692%
Relative lift      : 43.09%


In [10]:
practical_threshold = 0.005  # 0.50 percentage points; analytical assumption
alpha = 0.05

p_value = conversion_test["p_value"]
ci_lower = conversion_test["ci_lower"]
ci_upper = conversion_test["ci_upper"]
cohens_h = cohens_h(ad_rate, psa_rate)

print(f"Alpha: {alpha}")
print(f"Practical threshold: {practical_threshold:.2%}")
print(f"P-value: {p_value:.6g}")
print(f"95% CI: {ci_lower:.4%} to {ci_upper:.4%}")
print(f"Cohen's h: {cohens_h:.4f}")


Alpha: 0.05
Practical threshold: 0.50%
P-value: 1.70528e-13
95% CI: 0.5951% to 0.9434%
Cohen's h: 0.0530


In [11]:
statistically_significant = p_value < alpha
practically_significant = absolute_difference >= practical_threshold
ci_supports_positive_effect = ci_lower > 0

print(f"Statistically significant: {statistically_significant}")
print(f"Exceeds practical threshold: {practically_significant}")
print(f"95% CI entirely above zero: {ci_supports_positive_effect}")

Statistically significant: True
Exceeds practical threshold: True
95% CI entirely above zero: True


In [12]:
# Experiment validity assessment using the reusable validation functions from src/.

group_check = check_group_integrity(df)
user_check = check_user_uniqueness(df)
conversion_check = check_conversion_integrity(df)
srm_result = calculate_srm(df, assumed_allocation={"ad": 0.96, "psa": 0.04})

srm_p_value = srm_result["p_value"]
srm_ok = not srm_result["significant_srm"]
duplicates_ok = user_check["valid"]
missing_ok = conversion_check["valid"]
groups_ok = group_check["valid"]

print(f"SRM check passed: {srm_ok}")
print(f"Duplicate users check passed: {duplicates_ok}")
print(f"Missing conversion values check passed: {missing_ok}")
print(f"Treatment/control integrity check passed: {groups_ok}")
print(f"SRM p-value: {srm_p_value:.6g}")

print("\nImportant caveat:")
print("Randomization and intended allocation cannot be independently verified from the dataset.")


SRM check passed: True
Duplicate users check passed: True
Missing conversion values check passed: True
Treatment/control integrity check passed: True
SRM p-value: 0.999788

Important caveat:
Randomization and intended allocation cannot be independently verified from the dataset.


In [13]:
major_validity_issue = not (
    srm_ok
    and duplicates_ok
    and missing_ok
    and groups_ok
)

if (
    statistically_significant
    and practically_significant
    and ci_supports_positive_effect
    and not major_validity_issue
):
    decision = "SHIP"
elif not statistically_significant:
    decision = "NEED MORE DATA"
elif statistically_significant and not practically_significant:
    decision = "DO NOT SHIP"
else:
    decision = "NEED MORE DATA"

print(f"Major validity issue: {major_validity_issue}")
print(f"Decision: {decision}")


Major validity issue: False
Decision: SHIP


In [14]:
# Final decision audit: all decision inputs are calculated from the cleaned dataset.
decision_audit = pd.Series({
    "decision": decision,
    "ad_conversion_rate": ad_rate,
    "psa_conversion_rate": psa_rate,
    "absolute_difference": absolute_difference,
    "p_value": p_value,
    "ci_lower": ci_lower,
    "ci_upper": ci_upper,
    "cohens_h": cohens_h,
    "srm_p_value": srm_p_value,
    "major_validity_issue": major_validity_issue,
})

display(decision_audit.to_frame("value"))


,value
decision,SHIP
ad_conversion_rate,0.025547
psa_conversion_rate,0.017854
absolute_difference,0.007692
p_value,0.0
ci_lower,0.005951
ci_upper,0.009434
cohens_h,0.053003
srm_p_value,0.999788
major_validity_issue,False


In [15]:
# Illustrative business impact scenario
# These are assumptions, NOT observed values from the dataset.

eligible_users = 100_000

expected_conversions_psa = eligible_users * psa_rate
expected_conversions_ad = eligible_users * ad_rate

incremental_conversions = (
    expected_conversions_ad - expected_conversions_psa
)

print(f"Illustrative eligible users: {eligible_users:,}")
print(f"Expected conversions with PSA: {expected_conversions_psa:,.0f}")
print(f"Expected conversions with Ad: {expected_conversions_ad:,.0f}")
print(f"Illustrative incremental conversions: {incremental_conversions:,.0f}")

Illustrative eligible users: 100,000
Expected conversions with PSA: 1,785
Expected conversions with Ad: 2,555
Illustrative incremental conversions: 769


## Final Business Recommendation

### Decision: SHIP

Based on the available experimental evidence, the Ad variant should be considered for rollout, subject to confirmation of business economics and the experiment's randomization assumptions.

### Evidence Supporting the Decision

The Ad group achieved a conversion rate of **2.5547%**, compared with **1.7854%** for the PSA group.

This represents:

- **+0.7692 percentage points** absolute conversion lift
- **+43.09%** relative lift
- **95% CI: +0.5951 to +0.9434 percentage points**
- **p-value: 1.71 × 10⁻¹³**
- **Cohen's h: 0.0530**, indicating a small standardized effect

The observed absolute lift exceeds the predefined analytical practical-significance threshold of **+0.50 percentage points**.

### Experiment Validity

The available structural validation checks did not identify major data-integrity problems:

- No duplicate users were detected.
- No missing conversion values were detected.
- Treatment and control groups contained valid categories.
- No statistically significant Sample Ratio Mismatch was detected relative to the assumed 96% Ad / 4% PSA allocation.

However, the intended allocation and randomization process cannot be independently verified from the available dataset. Therefore, causal interpretation should remain conditional on the experiment having been properly randomized.

### Segment Findings

The positive conversion difference was observed across all seven days.

After Benjamini–Hochberg correction:

- Five of seven day-level comparisons remained statistically significant.
- Thursday and Sunday did not remain statistically significant.
- Only the 14:00 hourly segment remained statistically significant after correction across 24 hourly comparisons.

Segment results are treated as exploratory because day and hour represent observed behavioral/exposure characteristics rather than independently randomized pre-treatment characteristics.

### Business Impact Illustration

The dataset does not contain revenue, advertising cost, profit margin, AOV, or customer lifetime value.

Therefore, actual ROI or incremental revenue cannot be calculated from this dataset.

As an illustrative scenario, applying the observed conversion rates to 100,000 eligible users would produce approximately:

- PSA: **1,785 conversions**
- Ad: **2,555 conversions**
- Incremental conversions: **~769**

This is a scenario based on observed conversion rates and an assumed audience size, not observed business revenue.

### Recommendation

**SHIP the Ad variant for broader use, provided that the real-world campaign economics support the additional conversions and the experimental randomization is confirmed.**

Before full rollout, validate:

1. Incremental revenue or profit per conversion
2. Advertising cost and campaign efficiency
3. Whether the experiment was genuinely randomized
4. Performance on future traffic
5. Whether the observed effect persists outside the analyzed sample

The statistical evidence supports the Ad variant, but the final business decision should incorporate economics that are unavailable in this dataset.